# Notebook 01：获取数据与数据库概览

目标：了解 Alchequant 的本地 SQLite 数据结构，并确认示例数据范围。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DB_PATH = ROOT / 'data' / 'stocks.db'
print(f'项目根目录: {ROOT}')
print(f'数据库: {DB_PATH}')

In [ ]:
import pandas as pd
from src.database import get_connection, get_stock_list, get_stock_count, get_date_range, get_stock_data

conn = get_connection(str(DB_PATH))
stocks = get_stock_list(conn)
print(f'成分股清单: {len(stocks)} 只')
print(f'已有日线数据: {get_stock_count(conn)} 只')
stocks.head(10)

In [ ]:
rows = []
for _, row in stocks.iterrows():
    start, end = get_date_range(conn, row['code'])
    if start:
        df = get_stock_data(conn, row['code'])
        rows.append({
            '代码': row['code'],
            '名称': row['name'],
            '记录数': len(df),
            '起始日期': start,
            '结束日期': end,
        })
coverage = pd.DataFrame(rows).sort_values('代码')
coverage.head(20)

In [ ]:
sample_code = coverage.iloc[0]['代码']
sample = get_stock_data(conn, sample_code)
print(sample_code, sample['date'].min(), sample['date'].max(), len(sample))
sample.head()

In [ ]:
conn.close()

小结：发布版默认使用 `data/stocks.db` 示例数据库，平台可以离线复现核心页面和报告。